In [ ]:
import requests
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
import spacy
from datetime import datetime

In [ ]:
nlp = spacy.load("de_core_news_md")
CLIMATE_KEYWORDS = [
    "klima", "klimawandel", "klimakrise", "erderwärmung",
    "globale erwärmung", "treibhauseffekt", "treibhausgas",
    "co2", "kohlendioxid", "emission", "emissionen",
    "klimaschutz", "klimapolitik", "klimaneutral",
    "dekarbonisierung",
    
    "klimaziel", "klimaziele", "co2-preis", "emissionshandel",
    "klimaschutzgesetz", "paris-abkommen", "pariser abkommen",
    "eu-klimapaket", "green deal", "klimaplan", "klimabericht", "ipcc",

    "energiewende", "erneuerbare energien", "windkraft",
    "solarenergie", "photovoltaik", "wasserstoff",
    "kohlekraft", "kohleausstieg", "atomkraft",
    "stromnetz", "netzausbau", "e-mobilität",
    "verbrennerverbot", "industrieemissionen",
    
    "hitzewelle", "dürre", "hochwasser", "überschwemmung",
    "starkregen", "waldbrand", "extremwetter",
    "gletscherschmelze", "meeresspiegel", "wasserknappheit",
    "klimafolgen", "artensterben",

    "klimarisiko", "nachhaltigkeit", "nachhaltige investitionen",
    "esg", "grüne finanzierung", "klimafonds",
    "energiepreise", "strompreis", "gaspreis",

    "landwirtschaft", "trockenheit", "ernteschäden",
    "umweltschutz", "biodiversität", "naturschutz",
    "ökosystem", "umweltpolitik",

    "fridays for future", "klimaaktivisten",
    "klimaprotest", "letzte generation", "klimabewegung",
    "klimadebatte"
]
SEARCH_API = "https://www.spiegel.de/services/sitesearch/search"
SEGMENTS = ",".join([
    "spon", "spon_paid",
    "spon_international",
    "mmo", "mmo_paid",
    "hbm", "hbm_paid",
    "elf", "elf_paid",
    "effi", "effi_paid"
])
MAX_PAGES = 100
PAGE_SIZE = 20
OUTPUT_CSV = "spiegel_scrapped.csv"
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}
SOURCE = "SPIEGEL"
LANGUAGE = "de"

In [ ]:
def clean_text(text):
    if not text or not isinstance(text, str):
        return None
    return re.sub(r"\s+", " ", text).strip()
def keyword_hits(text):
    text = str(text).lower()
    return sum(k in text for k in CLIMATE_KEYWORDS)
def extract_author(soup):
    tag = soup.select_one('a[href*="/impressum/autor"]')
    return tag.get_text(strip=True) if tag else None
def extract_published_date(soup):
    tag = soup.select_one("time.timeformat")
    return tag.get("datetime") if tag else None
def is_boilerplate(text):
    if not text:
        return True
    bad = [ "print-abo", "spiegel+", "digital-zugang", "jetzt weiterlesen", "abo abschließen"]
    return any(b in text.lower() for b in bad)
def analyze_text(text):
    doc = nlp(text)
    actors = sorted(set(ent.text for ent in doc.ents if ent.label_ in ["PER", "ORG"]))
    sentences = list(doc.sents)
    return actors, len(actors), len(sentences), len(text)
def classify_article(text):
    t = text.lower()
    if any(k in t for k in ["gesetz", "regierung", "bundestag", "politik", "eu"]):
        return "Climate_Policy"
    if any(k in t for k in ["studie", "forschung", "ipcc"]):
        return "Climate_Science"
    if any(k in t for k in ["energie", "wind", "solar", "strom"]):
        return "Energy_Transition"
    if any(k in t for k in ["wirtschaft", "industrie", "markt"]):
        return "Climate_Economy"
    if any(k in t for k in ["protest", "aktivisten", "demonstration"]):
        return "Climate_Activism"
    if any(k in t for k in ["hitzewelle", "dürre", "hochwasser", "waldbrand"]):
        return "Climate_Impact"
    return "Other_Climate"

In [ ]:
rows = []
seen_urls = set()

In [ ]:
for keyword in CLIMATE_KEYWORDS:
    print(f"\n Keyword: {keyword}")
    for page in range(1, MAX_PAGES + 1):
        params = {
            "q": keyword,
            "page": page,
            "page_size": PAGE_SIZE,
            "segments": SEGMENTS,
            "after": -2208988800,
            "before": int(datetime.now().timestamp())
        }
        r = requests.get(SEARCH_API, params=params, headers=HEADERS, timeout=15)
        data = r.json()
        items = data.get("results", [])
        if not items:
            break
        for item in items:
            url = item.get("url")
            if not url or url in seen_urls:
                continue
            seen_urls.add(url)
            title = clean_text(item.get("title"))
            intro = clean_text(item.get("teaser") or item.get("intro"))
            html = requests.get(url, headers=HEADERS, timeout=15).text
            soup = BeautifulSoup(html, "html.parser")
            body = soup.find("div", attrs={"data-area": "body"})
            if not body:
                continue
            paragraphs = [
                p.get_text().strip()
                for p in body.find_all("p")
                if len(p.get_text().strip()) > 60
            ]
            content = clean_text(" ".join(paragraphs))
            if not content or len(content) < 1000:
                continue
            if is_boilerplate(content):
                continue
            relevance = keyword_hits(f"{title} {intro} {content}")
            if relevance < 2:
                continue
            author = extract_author(soup)
            pub_date = extract_published_date(soup)
            is_premium = bool(soup.select_one("[data-area='Paywall']"))
            actors, actor_count, sent_count, length = analyze_text(content)
            article_class = classify_article(content)
            rows.append({
                "URL": url,
                "Source": SOURCE,
                "Language": LANGUAGE,
                "Published_Date": pub_date,
                "Keyword_Matched": keyword,
                "Article_Classification": article_class,
                "Headline": title,
                "Intro": intro,
                "Content": content,
                "Content_Length": length,
                "Sentence_Count": sent_count,
                "Actors": ", ".join(actors),
                "Actor_Count": actor_count,
                "Author": author
            })
            print("Saved")
            time.sleep(1)

In [ ]:
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"File saved: {OUTPUT_CSV}")
print(f"Total relevant SPIEGEL climate articles: {len(df)}")